In [13]:
import numpy as np

class SWCCDataset:
    def __init__(self, path, lat_range, lon_range):
        cols = (0, 2, 3, 8, 9, 16, 25, 26)
        dtypes = ["U60", "U20", "U20", "f8", "f8", "U30", "f8", "f8"]
        data = np.genfromtxt(path, delimiter=",", names=True, dtype=dtypes,
                              encoding="latin1", usecols=cols)
        lat, lon = data["latitude_decimal_degrees"], data["longitude_decimal_degrees"]
        mask = (lat >= lat_range[0]) & (lat <= lat_range[1]) & (lon >= lon_range[0]) & (lon <= lon_range[1])
        self.data = data[mask]

    def profiles(self):
        return np.unique(self.data["layer_id"])

    def get_curve(self, layer_id):
        rows = self.data[self.data["layer_id"] == layer_id]
        return RetentionCurve(rows["lab_head_m"], rows["lab_wrc"], layer_id)


class RetentionCurve:
    def __init__(self, head, wrc, layer_id=None):
        order = np.argsort(head)
        self.head = head[order]
        self.theta = wrc[order]
        self.layer_id = layer_id

    def __call__(self, h):
        i = np.argmin(np.abs(self.head - h))
        return self.theta[i]

    def __len__(self):
        return len(self.head)

In [15]:
ds = SWCCDataset("data_raw/WRC_dataset_surya_et_al_2021_final.csv", (-5, 13), (-82, -66))
curve = ds.get_curve(ds.profiles()[0])
print(len(curve), curve.head, curve.theta, curve(5.0))
# 6 [0.01 0.032 0.3 3.3 24.5 155.0] [0.71 0.71 0.695 0.64 0.455 0.404] 0.64

6 [1.00e-02 3.20e-02 3.00e-01 3.30e+00 2.45e+01 1.55e+02] [0.71  0.71  0.695 0.64  0.455 0.404] 0.64


In [16]:
class RetentionCurve:
    # ... (igual que antes: __init__, __call__, __len__) ...

    def water_capacity(self):
        h, theta = self.head, self.theta
        c = np.zeros(len(h))
        c[1:-1] = (theta[2:] - theta[:-2]) / (h[2:] - h[:-2])
        c[0] = (theta[1] - theta[0]) / (h[1] - h[0])
        c[-1] = (theta[-1] - theta[-2]) / (h[-1] - h[-2])
        return c